In [5]:
from dotenv import load_dotenv
from anthropic import Anthropic
from rich.pretty import pprint
import json
import anthropic.types as t

load_dotenv()

client = Anthropic()
model = "claude-opus-4-6"

In [6]:
client = Anthropic()

response: t.Message = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key information from this email: John Smith (john@example.com) is interested in our Enterprise plan and wants to schedule a demo for next Tuesday at 2pm.",
        }
    ],
    output_config={
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "email": {"type": "string"},
                    "plan_interest": {"type": "string"},
                    "demo_requested": {"type": "boolean"},
                },
                "required": ["name", "email", "plan_interest", "demo_requested"],
                "additionalProperties": False,
            },
        }
    },
)

valid_json = json.loads(response.content[0].text)
pprint(valid_json)

{'name': 'John Smith', 'email': 'john@example.com', 'plan_interest': 'Enterprise', 'demo_requested': True}

In [22]:
from pydantic import BaseModel
from anthropic import Anthropic


class ContactInfo(BaseModel):
    name: str
    email: str
    plan_interest: str
    demo_requested: bool

contact_info_json_schema = ContactInfo.model_json_schema()
contact_info_json_schema["additionalProperties"] = False

client = Anthropic()

response = client.messages.parse(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[
        {
            "role": "user",
            "content": "Extract the key information from this email: John Smith (john@example.com) is interested in our Enterprise plan and wants to schedule a demo for next Tuesday at 2pm.",
        }
    ],
    output_format=ContactInfo,
    # output_config={ # returned None for parsed_output field
    #     "format": {
    #         "type": "json_schema",
    #         "schema": contact_info_json_schema
    #     }
    # },
)

pprint(response)
# print(response.parsed_output.name)

ParsedMessage[TypeVar](
│   id='msg_01VkAhkmWMaZ1zqGqEnufHXU',
│   container=None,
│   content=[
│   │   ParsedTextBlock[TypeVar](
│   │   │   citations=None,
│   │   │   text='{"name":"John Smith","email":"john@example.com","plan_interest":"Enterprise","demo_requested":true}',
│   │   │   type='text',
│   │   │   parsed_output=ContactInfo(
│   │   │   │   name='John Smith',
│   │   │   │   email='john@example.com',
│   │   │   │   plan_interest='Enterprise',
│   │   │   │   demo_requested=True
│   │   │   )
│   │   )
│   ],
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=393,
│   │   output_tokens=44,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)

In [30]:
from anthropic import transform_schema
from pydantic import TypeAdapter

# First convert Pydantic model to JSON schema, then transform
schema = TypeAdapter(ContactInfo).json_schema()
schema = transform_schema(schema)
pprint(schema)
# Modify schema if needed
# schema["properties"]["custom_field"] = {"type": "string"}

response = client.messages.create(
    model="claude-opus-4-7",
    max_tokens=1024,
    messages=[{"role": "user", "content": "..."}],
    output_config={
        "format": {"type": "json_schema", "schema": schema},
    },
)

pprint(response)

{
│   'type': 'object',
│   'title': 'ContactInfo',
│   'properties': {
│   │   'name': {'type': 'string', 'title': 'Name'},
│   │   'email': {'type': 'string', 'title': 'Email'},
│   │   'plan_interest': {'type': 'string', 'title': 'Plan Interest'},
│   │   'demo_requested': {'type': 'boolean', 'title': 'Demo Requested'}
│   },
│   'additionalProperties': False,
│   'required': ['name', 'email', 'plan_interest', 'demo_requested']
}

Message(
│   id='msg_01C3gXrcooXopramZEHTJCy8',
│   container=None,
│   content=[
│   │   TextBlock(
│   │   │   citations=None,
│   │   │   text='{"name":"","email":"","plan_interest":"","demo_requested":false}',
│   │   │   type='text'
│   │   )
│   ],
│   model='claude-opus-4-7',
│   role='assistant',
│   stop_details=None,
│   stop_reason='end_turn',
│   stop_sequence=None,
│   type='message',
│   usage=Usage(
│   │   cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0),
│   │   cache_creation_input_tokens=0,
│   │   cache_read_input_tokens=0,
│   │   inference_geo='global',
│   │   input_tokens=339,
│   │   output_tokens=30,
│   │   server_tool_use=None,
│   │   service_tier='standard'
│   )
)